<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabianalexisbernabecruz-code/portafolio-actuaria-unam/blob/main/08_Statistical_Learning_Supervised.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/fabianalexisbernabecruz-code/portafolio-actuaria-unam)

</div>



<font face="Times New Roman">

 <img src="https://upload.wikimedia.org/wikipedia/commons/c/ca/Escudo-UNAM-escalable.svg" width="150" height="170" align="left"/>

  <img src="https://upload.wikimedia.org/wikipedia/commons/5/5b/Escudo-facultad-ciencias-unam-escalable.svg" width="150" height="170" align="right"/>

<h5 align="center"><i> </i></h5>
<h1 align="center"><b>Aprendizaje Supervisado — Estudio de Caso</b></h1>
</font>
<font face="Times New Roman">

---
---

<h2 align="center"><i>Pipeline Completo de Clasificación y Evaluación</i></h2>
<h5 align="center"><i>Análisis exploratorio (EDA), imputación, transformaciones, entrenamiento de clasificadores y curvas ROC.</i></h5>

---
---

<h5 align="center"><i>Universidad Nacional Autónoma de México</i></h5>
<h5 align="center"><i>Facultad de Ciencias — Actuaría</i></h5>
<h5 align="center"><i>Materia: Sistemas de Estadística II</i></h5>

---
---

<font color="Black" face="Times New Roman">
<h5 align="center"><i>Bernabe Cruz Fabian Alexis</i></h5>

</font>
</font>

<font face="Times New Roman">

## Instrucciones de la Tarea

</font>


<font face="Times New Roman">

**Fuente:** Portal de Datos Abiertos de la Ciudad de México — Centro de Comando C5

**URL:** https://archivo.datos.cdmx.gob.mx/C5/incidentes_viales/inViales_2022_2024.csv

**Periodo:** 2022 – 2024 (504,261 registros crudos × 17 columnas).

**Variable respuesta:** `requirio_despacho` — binaria. Vale 1 si `codigo_cierre = 'A'` (Afirmativo) y 0 en otro caso (`D`, `F`, `I`, `N`).

**Tasa positiva observada:** 36.51% (73,016 / 200,000 tras muestreo estratificado).

**Pipeline:**
1. Regresión logística (MLG, familia Bernoulli con enlace logit).
2. Árboles de decisión binarios.
3. Máquinas de Soporte Vectorial (SVM).
4. K-Vecinos más Cercanos (KNN).
5. Redes Neuronales — MLP shallow.

</font>


<font face="Times New Roman">

### Parte 0 — Carga y primer vistazo ``


- Dimensiones, tipos de variables, head/tail
- Reporte de valores faltantes: conteo, porcentaje y patrón (MCAR/MAR/MNAR con justificación)

### Parte 1 — Análisis Exploratorio de Datos (EDA) ``

- Distribución de la variable respuesta con prueba estadística formal (Shapiro-Wilk si n<5000, D'Agostino-Pearson si n≥5000; o test de proporciones para clasificación)
- Histogramas + densidades para variables continuas; barplots para categóricas
- Matriz de correlación (Pearson para continuas, Cramér's V para categóricas, correlación biserial punto para mixtas)
- Detección de outliers: método IQR + z-score; reporte de porcentaje afectado por variable
- Para clasificación: gráficas de tasa de evento por categoría de cada feature (weight of evidence visual)
- **Markdown obligatorio:** Interpretación actuarial de al menos tres hallazgos del EDA

### Parte 2 — Feature Engineering y Transformaciones ``

- Imputación justificada (mediana/moda para MCAR; KNN-impute o modelo para MAR)
- Transformaciones de variables continuas: evaluar log, Box-Cox y Yeo-Johnson; justificar con prueba de normalidad post-transformación
- Creación de **mínimo 2 features derivadas** con justificación actuarial explícita (e.g., para Dataset 1: `duracion_respuesta = hora_cierre - hora_creacion`; para Dataset 3: `monto_por_edad = monto / edad_media_rango`)
- Encoding de categóricas:
  - Para clasificación binaria: Weight of Evidence (WoE) o target encoding
  - Para multiclase: OHE o encoding ordinal justificado
  - **Nota R:** demostrar cómo R maneja factores nativamente vs el proceso manual en Python
- Estandarización: aplicar **solo** donde el modelo lo requiere (SVM, KNN, Redes Neuronales); **no** en árboles — justificar en Markdown por qué

### Parte 3 — Partición y Validación Cruzada ``

- Comparar al menos **tres proporciones** de train/test (70/30, 75/25, 80/20); reportar varianza de métricas entre splits
- Para desbalance (Dataset 1 y 2): aplicar SMOTE y `class_weight='balanced'`; comparar AUC-ROC y F1-macro antes y después
- Implementar CV estratificado con k=5 y k=10; reportar media ± desviación estándar de la métrica principal
- **Markdown obligatorio:** demostrar algebraicamente por qué CV reduce el sesgo de estimación del error de generalización

### Parte 4 — Modelos `[50 pts, 10 por modelo]`

</font>


<font face="Times New Roman">

### Filosofía de selección del mejor modelo

Para cada método (árbol, SVM, KNN, MLP, MLG) deben construir una **función paramétrica** al estilo de `analizar_arbol()` / `pipeline_experimentos()` vistas en clase. La función recibe como argumentos el espacio de búsqueda (proporciones de split, hiperparámetros, transformaciones, estrategia de balanceo) y genera el **producto cartesiano** de configuraciones, entrena cada una con CV estratificado y acumula los resultados en un DataFrame. Al final selecciona automáticamente la configuración ganadora según una métrica compuesta:

$$\lambda_{\text{score}} = (1 - \lambda) \cdot \text{métrica\_cv} - \lambda \cdot |\text{métrica\_train} - \text{métrica\_cv}|, \quad \lambda \in [0,1]$$

donde $\lambda$ penaliza el overfitting. El **modelo ganador de cada método** es el que maximiza $\lambda_{\text{score}}$ con el $\lambda$ que el equipo justifique actuarialmente. Ese ganador es el único que pasa a la **Parte 5 — Comparativa Total**.

La función debe reportar una tabla con todas las configuraciones exploradas. Ejemplo del formato esperado para cada método:

**Tabla interna por método — Clasificación Binaria (ejemplo: SVM)**

| `kernel` | `C` | `gamma` | `prop_train` | `transf_X` | `balanceo` | AUC-train | Acc-train | AUC-CV (media±std) | AUC-test | Acc-test | F1-test | t (s) | $\lambda_{0.5}$-score | **Ganador** |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| lineal | 0.01 | — | 0.75 | none | none | 0.793 | 0.761 | 0.781 ± 0.032 | 0.769 | 0.742 | 0.701 | 1.2 | 0.775 | |
| lineal | 1.0 | — | 0.75 | std | smote | 0.841 | 0.809 | 0.823 ± 0.018 | 0.819 | 0.791 | 0.774 | 1.8 | 0.821 | |
| rbf | 1.0 | 0.1 | 0.80 | std | smote | 0.863 | 0.829 | 0.851 ± 0.021 | 0.847 | 0.812 | 0.798 | 4.3 | 0.849 | ✓ |
| rbf | 10.0 | 0.1 | 0.80 | std | smote | 0.971 | 0.948 | 0.862 ± 0.041 | 0.831 | 0.803 | 0.789 | 6.1 | 0.827 | |
| polinomial | 3 | — | 0.70 | std | class_weight | 0.819 | 0.784 | 0.798 ± 0.029 | 0.784 | 0.761 | 0.743 | 2.9 | 0.791 | |

> Nótese la fila 4: AUC-train=0.971 vs AUC-CV=0.862 — gap de overfitting de 0.109 que el $\lambda$-score penaliza correctamente, impidiendo que sea el ganador a pesar de tener el AUC-CV más alto.

**Tabla interna por método — Regresión (ejemplo: KNN)**

| `k` | `distancia` | `pesos` | `prop_train` | `transf_X` | `transf_y` | RMSE-train | R²-train | RMSE-CV (media±std) | RMSE-test | MAE-test | R²-test | t (s) | $\lambda_{0.5}$-score | **Ganador** |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 3 | euclidiana | uniform | 0.75 | none | none | 198,300 | 0.821 | 312,450 ± 28,100 | 324,800 | 210,300 | 0.581 | 0.4 | −318,625 | |
| 7 | euclidiana | distance | 0.75 | std | log | 181,200 | 0.849 | 198,320 ± 15,600 | 201,750 | 143,200 | 0.741 | 0.5 | −200,035 | |
| 11 | manhattan | distance | 0.80 | std | log | 183,100 | 0.845 | 187,640 ± 12,300 | 189,420 | 131,500 | 0.763 | 0.6 | **−188,530** | ✓ |
| 15 | minkowski | uniform | 0.80 | std | log | 179,800 | 0.853 | 193,210 ± 14,800 | 196,880 | 138,700 | 0.751 | 0.7 | −195,045 | |
| 21 | euclidiana | distance | 0.80 | std | yeo-johnson | 186,400 | 0.839 | 201,330 ± 18,200 | 205,610 | 147,900 | 0.731 | 0.8 | −203,470 | |

> **Nota:** para regresión el $\lambda_{\text{score}}$ se expresa en unidades negativas de RMSE (mayor = mejor). Incluir al menos 15 configuraciones exploradas por método; la tabla de ejemplo muestra solo 5 filas para ilustrar el formato.

</font>


<font face="Times New Roman">

Una vez seleccionado el ganador de cada método, compiten en una sola tabla. Ejemplo del formato esperado:

**Dataset 1 — Clasificación Binaria: Incidentes Viales C5**

| Método | Mejor configuración | `transf_X` | `transf_y` | RMSE-train | R²-train | RMSE-CV (media±std) |
|---|---|---|---|---|---|---|
| MLG Gamma | Tweedie p=1.5 | std | log | 181,200 | 0.851 | 194,100 ± 11,200 |
| Árbol | depth=5, λ=0.5 | none | log | 176,300 | 0.861 | 208,320 ± 14,600 |
| SVR | rbf, C=10, ε=0.1 | std | log | 179,800 | 0.854 | 199,840 ± 12,900 |
| **KNN** | **k=11, manhattan** | **std** | **log** | **183,100** | **0.845** | **192,010 ± 10,400** |
| MLP | (32,), relu, α=1e-3 | std | log | 182,400 | 0.847 | 197,340 ± 11,800 |

*Evaluación en test y selección:*

| Método | RMSE-test | MAE-test | R²-test | t (s) | $\lambda_{0.5}$-score |
|---|---|---|---|---|---|
| MLG Gamma | 189,300 | 128,400 | 0.771 | 0.4 | −191,700 |
| Árbol | 201,750 | 139,100 | 0.748 | 0.9 | −205,035 |
| SVR | 195,420 | 133,800 | 0.760 | 18.3 | −197,630 |
| **KNN** | **187,640** | **131,500** | **0.763** | **0.6** | **−188,530** |
| MLP | 193,870 | 132,900 | 0.762 | 9.7 | −195,605 |

> El modelo en **negritas** es el ganador de cada comparativa. La discusión de la Parte 5 debe justificar la elección considerando no solo la métrica principal sino también el gap train/CV (proxy de overfitting), pérdida esperada, tiempo de cómputo e interpretabilidad actuarial.

</font>


<font face="Times New Roman">

Para cada modelo reportar adicionalmente una gráfica diagnóstica específica (detallada en cada subsección).

#### 4.1 Regresión / MLG

- **Clasificación binaria:** Regresión logística — justificar función de enlace logit; reporte de odds-ratios con IC al 95%
- **Clasificación multiclase:** Regresión logística multinomial (softmax); reporte de probabilidades por clase
- **Regresión:** MLG con familia Gamma o Tweedie — justificar elección de familia y función de enlace (log); comparar con OLS sobre variable transformada; interpretar coeficientes en escala original
- **Markdown obligatorio:** demostración de que la log-verosimilitud de la familia Gamma pertenece a la familia exponencial

#### 4.2 Árboles de Decisión

- Comparar criterio de Gini vs entropía (clasificación) o MSE vs MAE (regresión)
- Búsqueda de hiperparámetros: `minsplit/min_samples_split`, `maxdepth`, `cp` — usar GridSearchCV (Python) y `expand.grid` + loop (R)
- Visualización del árbol podado + importancia de variables (Python y R)
- **Markdown obligatorio:** demostración de que minimizar la impureza de Gini es equivalente a maximizar la ganancia de información para clasificación binaria

#### 4.3 SVM / SVR

- **Clasificación:** Kernels lineal, polinomial y RBF; CV en C y γ mediante GridSearchCV
- **Regresión:** SVR con kernel RBF; CV en C, ε y γ
- Visualización de frontera de decisión con PCA a 2D si p > 2
- Reporte de número de vectores soporte y su interpretación
- **Markdown obligatorio:** justificación del problema dual de optimización y por qué el kernel trick evita la maldición de la dimensionalidad

#### 4.4 KNN

- Curva de error de validación vs k (k = 1, 3, 5, 7, …, 31)
- Comparar distancias: Euclidiana, Manhattan, Minkowski con p=3
- Para clasificación: comparar pesos uniformes vs ponderados por distancia
- **Markdown obligatorio:** demostrar que KNN-1 tiene error de Bayes asintótico ≤ 2 × error de Bayes óptimo

#### 4.5 Redes Neuronales — MLP Shallow (1–2 capas ocultas)

- Implementación **exclusivamente con scikit-learn**: `MLPClassifier` (clasificación) y `MLPRegressor` (regresión); se puede usar keras/tensorflow
- Arquitectura: 1 capa oculta como base; como variante opcional una segunda — justificar número de neuronas con regla de pulgar $h \in [p, 2p]$ donde $p$ es el número de features
- Funciones de activación a comparar: `relu` vs `tanh`; la capa de salida la maneja sklearn internamente
- Hiperparámetros a buscar con `GridSearchCV`: `hidden_layer_sizes`, `activation`, `alpha` (L2), `learning_rate_init`
- Reportar `loss_curve_` para verificar convergencia; si no converge en `max_iter=500`, reportarlo y justificar
- Implementación en R: paquete `nnet` (una capa oculta, parámetro `size`)
- **Markdown obligatorio:**
  - Enunciado del teorema de aproximación universal (sin demostración completa): un MLP con una capa oculta y activación sigmoide aproxima cualquier función continua en un compacto de $\mathbb{R}^n$
  - Esquema de backpropagation para una sola capa oculta: mostrar la expresión de $\partial \mathcal{L}/\partial W^{(1)}$ via regla de la cadena — no se pide el caso general

### Parte 5 — Comparación y Conclusiones ``

- Tabla resumen con todas las métricas (AUC-ROC/F1-macro para clasificación; RMSE/MAE/R² para regresión) — train y test
- Prueba estadística de comparación de modelos:
  - Clasificación: test de DeLong para diferencia de AUCs
  - Regresión: test de Diebold-Mariano
- Análisis de costo de error: definir matriz de costos actuarialmente motivada y reportar pérdida esperada por modelo (no solo accuracy)
- Análisis de sensibilidad del threshold en clasificación binaria: curva ROC + criterio de Youden para threshold óptimo
- Discusión final (Markdown): ¿qué modelo recomendarían para producción desde una perspectiva actuarial? Argumentar con base en interpretabilidad, costo computacional y riesgo de sobreajuste


## Criterios de penalización y bonificación

| Situación | Efecto |
|---|---|
| Usar solo `accuracy` en dataset desbalanceado sin justificación | −5 pts |
| Estandarizar features para árbol sin justificar que es innecesario | −3 pts |
| Código que no corre en Colab | −10 pts automáticos |
| No incluir Markdown teórico entre secciones | −5 pts |
| Implementar **un modelo desde cero** (sin sklearn/caret/keras): KNN, árbol o MLP | +5 pts |
| Análisis de calibración de probabilidades (Platt scaling o isotonic) para todos los clasificadores | +3 pts |
| Comparación de tiempo de cómputo y discusión de trade-off práctico (incluir red neuronal) | +2 pts |
| Visualización de pesos de la primera capa del MLP como heatmap de importancia | +2 pts |



## Notas técnicas

1. **Python y R:** cada sección de modelado debe tener implementación en ambos lenguajes. Las celdas de R se ejecutan con `%%R` vía `rpy2`.
2. **Gestión de memoria:** liberar objetos grandes con `del` + `gc.collect()` en Python y `rm()` + `gc()` en R. Los notebooks no deben consumir más de 12 GB de RAM en Colab.
3. **Reproducibilidad:** fijar semillas (`random_state=42` en Python, `set.seed(42)` en R) en todas las operaciones estocásticas.
4. **Descarga de datos:** implementar descarga programática desde las URLs oficiales; no incluir archivos CSV en el repositorio.

</font>


<font face="Times New Roman">

## Setup

</font>


<font face="Times New Roman">

Aquí se importan todas las librerías de Python que se utilizarán a lo largo del notebook para el preprocesamiento de datos, modelado, evaluación y visualización. También se configuran algunas opciones globales y se fija la semilla aleatoria para asegurar la reproducibilidad.

</font>


In [ ]:
import os, time, gc, io, json, warnings, itertools, pathlib
from urllib.request import urlopen, Request
from urllib.error import URLError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_curve, brier_score_loss, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print('imbalanced-learn no disponible; SMOTE se omite.')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 90
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Python stack listo.')

In [ ]:
# Activacion rpy2
try:
    import rpy2
    from rpy2.robjects import r as R_raw
    try:
        get_ipython().run_line_magic('load_ext', 'rpy2.ipython')
        print('rpy2 %s cargado correctamente.' % rpy2.__version__)
    except Exception as e:
        print('rpy2 importado pero magic no disponible:', e)
    R_raw('set.seed(42)')
except ImportError:
    print('rpy2 NO disponible. Las celdas %%R se saltaran.')

<font face="Times New Roman">

## Parte 0 - Carga y primer vistazo ``

Antes de modelar es necesario:
1. Validar que los tipos de datos son coherentes con el diagrama de variables del portal.
2. Cuantificar valores faltantes y clasificar su patron:
   - **MCAR** (Missing Completely At Random): $P(R \mid X_{obs}, X_{mis}) = P(R)$.
   - **MAR** (Missing At Random): $P(R \mid X_{obs}, X_{mis}) = P(R \mid X_{obs})$.
   - **MNAR** (Missing Not At Random): la probabilidad de faltante depende del valor no observado.

La distincion importa porque la imputacion simple es **insesgada solo bajo MCAR**; bajo MAR se requiere imputacion condicional (KNN-impute, MICE); bajo MNAR se necesita modelar el mecanismo.

</font>


In [ ]:
import pandas as pd
import numpy as np
import os

DATA_URL = "https://archivo.datos.cdmx.gob.mx/C5/incidentes_viales/inViales_2022_2024.csv"
FALLBACK_URL = "https://drive.google.com/file/d/1RP2U0tCu4kf4JRmUzeAHXhZefT1qdCTO/view?usp=sharing"
LOCAL_CSV = "inViales_2022_2024.csv"

ARCHIVO_TEMP = "inViales_temp.csv"
RANDOM_STATE = 42

def _muestreo_estratificado(df, col, n, rs):
    """Muestreo proporcional por grupo conservando todas las columnas."""
    partes = []
    total = len(df)
    for _, g in df.groupby(col, sort=False):
        k = max(1, int(n * len(g) / total))
        partes.append(g.sample(min(len(g), k), random_state=rs))
    return pd.concat(partes).sample(frac=1, random_state=rs).reset_index(drop=True)

def cargar_c5(nrows=None, sample_n=200_000):
    df = None

    # 1) URL primaria (portal CDMX)
    try:
        print(f"Descargando: {DATA_URL}")
        df = pd.read_csv(DATA_URL, nrows=nrows, low_memory=False,
                         encoding="utf-8", on_bad_lines="skip")
        print("   Descarga primaria OK.")
    except Exception as e:
        print(f"   URL primaria fallo: {e}")

    # 2) Drive (mismo archivo, mirror)
    if df is None:
        try:
            import gdown
            print(f"Intentando fallback Drive: {FALLBACK_URL}")
            gdown.download(FALLBACK_URL, ARCHIVO_TEMP, quiet=False, fuzzy=True)
            df = pd.read_csv(ARCHIVO_TEMP, nrows=nrows, low_memory=False,
                             encoding="utf-8", on_bad_lines="skip")
            print("   Descarga Drive OK.")
        except Exception as e:
            print(f"   Drive fallo: {e}")

    print(f"  shape crudo: {df.shape}")
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

    if sample_n and len(df) > sample_n:
        col = next((c for c in df.columns if "cierre" in c and "codigo" in c), None)
        if col:
            df = _muestreo_estratificado(df, col, sample_n, RANDOM_STATE)
        else:
            df = df.sample(sample_n, random_state=RANDOM_STATE).reset_index(drop=True)

    if os.path.exists(ARCHIVO_TEMP):
        try: os.remove(ARCHIVO_TEMP)
        except: pass
    return df

if __name__ == "__main__":
    df = cargar_c5(sample_n=200_000)
    print("DataFrame final listo:", df.shape)



<font face="Times New Roman">

Estas líneas muestran información básica del DataFrame cargado, incluyendo sus dimensiones, tipos de datos (`df.info()`), y una vista previa de las primeras (`df.head()`) y últimas (`df.tail()`) filas.

</font>


In [ ]:
print('=== info ===')
df.info()
print('\n=== head ===')
display(df.head())
print('\n=== tail ===')
display(df.tail())

<font face="Times New Roman">

Se define la variable objetivo `y` como binaria (`0` o `1`) a partir de la columna `codigo_cierre`, donde 'A' indica despacho (1) y otras categorías (N, F, D) indican no despacho (0).

</font>


In [ ]:
def construir_target(df):
    col_cierre = next(c for c in df.columns if 'cierre' in c and 'codigo' in c)
    y = (df[col_cierre].astype(str).str.strip().str.upper() == 'A').astype(int)
    return y, col_cierre

y, col_cierre = construir_target(df)
print(f'Columna fuente: {col_cierre}')
print(f'Tasa base despacho: {y.mean():.4f}')
print(f'Distribucion codigo_cierre:\n{df[col_cierre].value_counts(dropna=False)}')

In [ ]:
def reporte_faltantes(df):
    na = df.isna().sum()
    pct = (na / len(df) * 100).round(3)
    out = pd.DataFrame({'n_missing': na, 'pct_missing': pct}).query('n_missing > 0')
    return out.sort_values('pct_missing', ascending=False)

rep = reporte_faltantes(df)
display(rep if len(rep) else pd.DataFrame({'msg':['sin faltantes']}))

<font face="Times New Roman">

La función `reporte_faltantes` cuenta los nulos por columna. Hay faltantes en cinco columnas, todas en proporciones bajas: `colonia_catalogo` (2.26%), `alcaldia_catalogo` (0.113%), `alcaldia_inicio` y `alcaldia_cierre` (0.006%) y `tipo_entrada` (0.001%). Como todas son geográficas o de captura, el patrón es plausiblemente **MAR** y la imputación por moda/mediana es razonable.

</font>


<font face="Times New Roman">

Esta celda muestra la correlación entre los indicadores de missingness.

</font>


In [ ]:
# Patron de faltantes: correlacion entre indicadores de missingness
miss = df.isna().astype(int)
miss = miss.loc[:, miss.sum() > 0]
if miss.shape[1] > 1:
    fig, ax = plt.subplots(figsize=(7,5))
    sns.heatmap(miss.corr(), cmap='mako', annot=True, fmt='.2f', ax=ax)
    ax.set_title('Correlacion entre indicadores de missingness')
    plt.tight_layout(); plt.show()
else:
    print('No hay columnas con faltantes suficientes para correlacion.')

<font face="Times New Roman">

## Parte 1 - Analisis Exploratorio de Datos ``

El EDA tiene dos objetivos: (i) identificar el soporte y la forma de la distribucion de cada variable para elegir transformaciones y funciones de enlace adecuadas; (ii) detectar estructuras (outliers, multimodalidad, correlaciones) que rompan los supuestos de los modelos. Para el target binario, la prueba de proporciones contrasta $H_0: p = 0.5$ contra una tasa base observada; si se rechaza, hay desbalance.

</font>


In [ ]:
n = len(y); p_hat = y.mean()
from statsmodels.stats.proportion import proportions_ztest
stat, pval = proportions_ztest(y.sum(), n, value=0.5)
print(f'n = {n}; p_hat = {p_hat:.4f}; H0: p=0.5; z={stat:.3f}; p-value={pval:.2e}')

<font face="Times New Roman">

El test de proporciones rechaza $H_0: p = 0.5$ con $\hat p = 0.3651$, $z = -125.32$ y p-value $\approx 0$. La magnitud del desbalance es **moderada**: el split es 63.49% / 36.51%. Aplicaremos las técnicas de balanceo (`class_weight='balanced'` y SMOTE).

</font>


In [ ]:
fig, ax = plt.subplots(figsize=(5.5,3.5))
y.value_counts().plot(kind='bar', color=['#c8553d','#588b8b'], ax=ax)
ax.set_xticklabels(['No despacho (0)','Despacho (1)'], rotation=0)
ax.set_title(f'Distribucion del target (tasa base = {p_hat:.3f})')
plt.tight_layout(); plt.show()

<font face="Times New Roman">

Este gráfico de barras visualiza la distribución de la variable objetivo, mostrando la proporción de incidentes que requirieron despacho (1) y los que no (0). La tasa base de despacho se indica para contextualizar el nivel de desbalance.

</font>


<font face="Times New Roman">

La función `parsear_tiempos` convierte las columnas de fecha y hora a formatos numéricos (`datetime64[ns]`, `dt.hour`) y calcula un feature, `duracion_min`, que representa la duración de cada incidente en minutos.

</font>


In [ ]:
def parsear_tiempos(df):
    df = df.copy()
    for c in [c for c in df.columns if "fecha" in c]:
        df[c] = pd.to_datetime(df[c], errors="coerce")

    for c in [c for c in df.columns if "hora" in c and "num" not in c]:
        s = df[c]
        is_text = s.dtype == object or s.dtype.kind in ("O","U","S") or (len(s) and isinstance(s.iloc[0], str))
        if is_text:
            df[c+"_num"] = pd.to_datetime(s.astype(str), format="%H:%M:%S", errors="coerce").dt.hour
    if "fecha_creacion" in df.columns and "fecha_cierre" in df.columns:
        df["duracion_min"] = (df["fecha_cierre"] - df["fecha_creacion"]).dt.total_seconds() / 60
        df.loc[df["duracion_min"] < 0, "duracion_min"] = np.nan

    if "alcaldia_hechos" not in df.columns and "alcaldia_inicio" in df.columns:
        df["alcaldia_hechos"] = df["alcaldia_inicio"]
    return df

df = parsear_tiempos(df)
print("Columnas:", list(df.columns))

<font face="Times New Roman">

Se generan histogramas de densidad para las variables continuas clave (`latitud`, `longitud`, `hora_creacion_num`, `duracion_min`) para visualizar sus distribuciones. Esto ayuda a identificar la forma, el rango y posibles asimetrías o la presencia de outliers en estas características numéricas.

</font>


In [ ]:
continuas = [c for c in ['latitud','longitud','hora_creacion_num','duracion_min'] if c in df.columns]
fig, axes = plt.subplots(2, 2, figsize=(11,7))
for ax, c in zip(axes.flat, continuas):
    sns.histplot(df[c].dropna(), kde=True, bins=40, ax=ax, color='#3d5a80')
    ax.set_title(c)
for ax in axes.flat[len(continuas):]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()

<font face="Times New Roman">

Se crean gráficos de barras para las principales variables categóricas, mostrando la frecuencia de ocurrencia de las 15 categorías más comunes. Esto permite observar la distribución y cardinalidad de variables como `alcaldia_hechos` e `incidente_c4`.

</font>


In [ ]:
cats = [c for c in ['alcaldia_hechos','incidente_c4','tipo_entrada','dia_semana','clas_con_f_alarma'] if c in df.columns]
fig, axes = plt.subplots(len(cats), 1, figsize=(11, 2.8*len(cats)))
if len(cats) == 1: axes = [axes]
for ax, c in zip(axes, cats):
    order = df[c].value_counts().head(15).index
    sns.countplot(data=df, y=c, order=order, ax=ax, color='#3d5a80')
    ax.set_title(f'{c} (top 15)')
plt.tight_layout(); plt.show()

<font face="Times New Roman">

Esta celda calcula y visualiza matrices de correlación: Pearson para variables continuas, Cramer's V para categóricas y correlación biserial-punto con el target para identificar relaciones con la variable respuesta binaria.

</font>


In [ ]:
# Matrices de correlacion
cont_exist = [c for c in continuas if df[c].notna().sum() > 0]
fig, ax = plt.subplots(figsize=(5.5,4))
sns.heatmap(df[cont_exist].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Pearson - continuas')
plt.tight_layout(); plt.show()

def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tab, correction=False)[0]
    n_ = tab.to_numpy().sum()
    r, k = tab.shape
    return np.sqrt(chi2 / (n_ * (min(r, k) - 1))) if min(r,k) > 1 else 0

cats_mat = [c for c in cats if c in df.columns]
if len(cats_mat) >= 2:
    mat = pd.DataFrame(index=cats_mat, columns=cats_mat, dtype=float)
    for a in cats_mat:
        for b in cats_mat:
            mat.loc[a, b] = cramers_v(df[a].astype(str), df[b].astype(str))
    fig, ax = plt.subplots(figsize=(6,5))
    sns.heatmap(mat.astype(float), annot=True, fmt='.2f', cmap='viridis', ax=ax)
    ax.set_title("Cramer's V - categoricas")
    plt.tight_layout(); plt.show()

from scipy.stats import pointbiserialr
print('Correlacion biserial-punto con target (requirio_despacho):')
for c in cont_exist:
    m = df[c].notna()
    r, p = pointbiserialr(y[m], df.loc[m, c])
    print(f'  {c:22s} r={r:+.3f}  p={p:.2e}')

<font face="Times New Roman">

La función `outliers_iqr_z` detecta valores atípicos utilizando los métodos de rango intercuartílico (IQR) y Z-score. Se reporta el porcentaje de datos identificados como outliers para cada variable continua, ofreciendo una perspectiva sobre la dispersión de los datos.

</font>


In [ ]:
def outliers_iqr_z(s):
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    iqr_pct = ((s<lo)|(s>hi)).mean() * 100
    z = np.abs((s - s.mean()) / s.std())
    z_pct = (z > 3).mean() * 100
    return {'iqr_pct': iqr_pct, 'z_pct': z_pct}

out = pd.DataFrame({c: outliers_iqr_z(df[c]) for c in cont_exist}).T
display(out.round(2))

<font face="Times New Roman">

Estos gráficos de barras muestran la tasa de despacho (`P(despacho | categoría)`) para las categorías más frecuentes de cada variable categórica. La línea punteada indica la tasa base global, facilitando la identificación de categorías con mayor o menor propensión a requerir despacho.

</font>


In [ ]:
df_tmp = df.assign(y=y.values)
fig, axes = plt.subplots(len(cats_mat), 1, figsize=(11, 2.6*len(cats_mat)))
if len(cats_mat) == 1: axes = [axes]
for ax, c in zip(axes, cats_mat):
    tasa = df_tmp.groupby(c)['y'].agg(['mean','count']).query('count >= 50')
    tasa = tasa.sort_values('mean', ascending=False).head(20)
    sns.barplot(x=tasa['mean'], y=tasa.index, ax=ax, color='#ee6c4d')
    ax.axvline(y.mean(), color='k', ls='--', lw=1, label=f'tasa base={y.mean():.2f}')
    ax.set_title(f'Tasa de despacho por {c}')
    ax.set_xlabel('P(despacho | categoria)')
    ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

<font face="Times New Roman">

### Interpretación de hallazgos

1.  **El motivo del incidente y el tipo de entrada muestran una clara relación con la tasa de despacho.** Los gráficos de barra de la tasa de despacho por categoría para `incidente_c4` y `tipo_entrada` revelan que ciertas categorías tienen una propensión mucho mayor a requerir despacho que otras, superando significativamente la tasa base. Esto sugiere que estas variables son fuertes indicadores de la necesidad de recursos de emergencia, un hallazgo importante para la estimación de tarifas de seguros.

2.  **Heterogeneidad geográfica débil pero detectable en las coordenadas y duración del incidente.** Las correlaciones biserial-punto con la variable respuesta son pequeñas pero estadísticamente significativas para `latitud` ($r=+0.050$, p=4.5e-112), `longitud` ($r=-0.012$, p=1.1e-07), `hora_creacion_num` ($r=-0.061$, p=1.1e-164) y `duracion_min` ($r=+0.067$, p=2.5e-197).

3.  **Desbalance moderado de clases con implicaciones para la evaluación.** La prueba de proporciones rechaza la hipótesis nula $H_0: p = 0.5$ con un p-value prácticamente cero y una tasa observada de despacho ($\hat p$) de 0.3651. Esto confirma un desbalance moderado, con aproximadamente 36.51% de incidentes requiriendo despacho y 63.49% no. Consecuentemente, un modelo trivial que predijera siempre "no despacho" lograría un 63.5% de exactitud, lo que resalta la necesidad de utilizar métricas de desempeño robustas que no se vean sesgadas por el desbalance de clases, como la curva ROC o F1-score.

4.  **Presencia notable de outliers en `duracion_min`.** El análisis de outliers revela que el método IQR detecta un 22.01% de valores atípicos en `duracion_min`, mientras que para `latitud`, `longitud` y `hora_creacion_num` este porcentaje es inferior al 1%. Esta elevada proporción de outliers en `duracion_min` sugiere una distribución con una cola muy larga, donde la mayoría de los incidentes se resuelven rápidamente, pero un subconjunto significativo tarda mucho más.

</font>


<font face="Times New Roman">

## Parte 2 - Feature Engineering y Transformaciones ``

Las transformaciones de variables buscan (i) estabilizar la varianza, (ii) aproximar simetria/gaussianidad cuando el modelo lo requiere (e.g. MLG Gaussiana, SVM con RBF), y (iii) capturar no-linealidades mediante features derivadas. Para arboles de decision cualquier transformacion monotona es invisible (el split depende solo del orden) por lo que no se estandarizan ni transforman las continuas. SVM/KNN/MLP son sensibles a la escala (distancias y activaciones saturables), por lo que si requieren `StandardScaler` o equivalente.

</font>


In [ ]:
def imputar(df, cols_num, cols_cat, metodo='mediana'):
    df = df.copy()
    if metodo == 'mediana':
        df[cols_num] = SimpleImputer(strategy='median').fit_transform(df[cols_num])
    elif metodo == 'knn':
        df[cols_num] = KNNImputer(n_neighbors=5).fit_transform(df[cols_num])
    for c in cols_cat:
        df[c] = df[c].fillna(df[c].mode().iloc[0] if df[c].notna().any() else 'DESCONOCIDO')
    return df

<font face="Times New Roman">

La función `imputar` rellena nulos numéricos con mediana o KNN, y categóricos con la moda. El dataset tiene faltantes en cinco columnas (todas geográficas, <2.3%), patrón MAR/MCAR, la imputación simple es suficiente.

</font>


<font face="Times New Roman">

`test_normalidad_pre_post` evalúa cómo afectan tres transformaciones (`log1p`, Box-Cox, Yeo-Johnson) la normalidad de una variable continua. Para `duracion_min` los cuatro p-values son efectivamente cero, D'Agostino-Pearson rechaza normalidad antes y después de cualquier transformación, lo cual confirma la asimetría extrema. Por eso dejamos `duracion_min` sin transformar.

</font>


In [ ]:
def test_normalidad_pre_post(s):
    s = s.dropna()
    if len(s) > 5000:
        _, p0 = stats.normaltest(s)
    else:
        _, p0 = stats.shapiro(s.sample(min(len(s), 4999), random_state=RANDOM_STATE))
    if (s > 0).all():
        s_log = np.log1p(s)
        s_bc, _ = stats.boxcox(s + 1e-6)
    else:
        s_log = s.copy(); s_bc = s.copy()
    s_yj = PowerTransformer(method='yeo-johnson').fit_transform(s.values.reshape(-1,1)).ravel()
    def p_(x):
        x = pd.Series(x).dropna()
        if len(x) > 5000: return stats.normaltest(x).pvalue
        return stats.shapiro(x.sample(min(len(x),4999), random_state=42)).pvalue
    return {'orig': p0, 'log1p': p_(s_log), 'boxcox': p_(s_bc), 'yeo_johnson': p_(s_yj)}

if 'duracion_min' in df.columns:
    res = test_normalidad_pre_post(df['duracion_min'])
    print('p-values de normalidad (mayor = mas normal) para duracion_min:')
    for k, v in res.items():
        print(f'  {k:14s}  p = {v:.3e}')

<font face="Times New Roman">

`construir_features` agrega las features derivadas: componentes cíclicos `hora_sin`/`hora_cos` (encoding trigonométrico de la hora, así 23h y 0h quedan vecinas), día de la semana (`dow`), bandera `es_fin_semana`, mes y `densidad_alcaldia` (proporción relativa de incidentes por alcaldía).

</font>


In [ ]:
def construir_features(df):
    df = df.copy()
    if 'hora_creacion_num' in df.columns:
        h = df['hora_creacion_num'].fillna(12)
        df['hora_sin'] = np.sin(2*np.pi*h/24)
        df['hora_cos'] = np.cos(2*np.pi*h/24)
    if 'fecha_creacion' in df.columns:
        df['dow'] = df['fecha_creacion'].dt.dayofweek
        df['es_fin_semana'] = (df['dow'] >= 5).astype(int)
        df['mes'] = df['fecha_creacion'].dt.month
    if 'alcaldia_hechos' in df.columns:
        cnt = df['alcaldia_hechos'].value_counts()
        df['densidad_alcaldia'] = df['alcaldia_hechos'].map(cnt) / len(df)
    return df

df = construir_features(df)

<font face="Times New Roman">

La función `woe_encode` implementa el cálculo de Weight of Evidence (WoE) para variables categóricas. Esta técnica codifica las categorías basándose en la relación con la variable objetivo.

</font>


In [ ]:
def woe_encode(series, y, laplace=0.5):
    df_ = pd.DataFrame({'x': series.astype(str), 'y': y})
    grp = df_.groupby('x')['y'].agg(['sum','count'])
    grp['bad'] = grp['sum']; grp['good'] = grp['count'] - grp['sum']
    total_bad = grp['bad'].sum(); total_good = grp['good'].sum()
    grp['woe'] = np.log(((grp['bad']+laplace)/total_bad) /
                        ((grp['good']+laplace)/total_good))
    return series.astype(str).map(grp['woe']).fillna(0), grp['woe'].to_dict()

woe_maps = {}
for c in ['incidente_c4','alcaldia_hechos','tipo_entrada','clas_con_f_alarma']:
    if c in df.columns:
        df[c+'_woe'], woe_maps[c] = woe_encode(df[c], y)
print('Variables WoE creadas:', [c+'_woe' for c in woe_maps])

<font face="Times New Roman">

Aquí se selecciona el conjunto final de **14 características** para el modelado: las cuatro continuas originales (`latitud`, `longitud`, `hora_creacion_num`, `duracion_min`), las seis derivadas (`hora_sin`, `hora_cos`, `dow`, `es_fin_semana`, `mes`, `densidad_alcaldia`) y las cuatro WoE de las categóricas (`incidente_c4`, `alcaldia_hechos`, `tipo_entrada`, `clas_con_f_alarma`). Se imputan los faltantes restantes con mediana, dando una matriz `X` de **199,998 × 14** con tasa positiva 0.365.

</font>


In [ ]:
FEATURES = [
    'latitud','longitud','hora_creacion_num','duracion_min',
    'hora_sin','hora_cos','dow','es_fin_semana','mes','densidad_alcaldia',
    'incidente_c4_woe','alcaldia_hechos_woe','tipo_entrada_woe','clas_con_f_alarma_woe',
]
FEATURES = [f for f in FEATURES if f in df.columns]

for c in FEATURES:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

X = df[FEATURES].copy()
print(f'X: {X.shape}  |  y: {y.shape}  |  tasa positiva: {y.mean():.3f}')
X.head()

<font face="Times New Roman">

## Parte 3 - Particion y Validacion Cruzada ``

La validacion cruzada con $k$ folds estima el error de generalizacion como $\hat R_{CV} = \frac{1}{k}\sum_{i=1}^{k} \hat L(f^{(-i)}, D_i)$ donde $f^{(-i)}$ es el modelo entrenado excluyendo el fold $i$. Bajo la hipotesis de i.i.d. y $k \to n$ (LOO), $\mathbb{E}[\hat R_{CV}] \to R(f)$ de manera casi insesgada (sesgo $O(1/n)$). Para $k$ finito existe un trade-off: $k$ pequeno reduce varianza pero introduce sesgo pesimista (menor tamaño de entrenamiento); $k$ grande reduce sesgo pero los folds se correlacionan mas, aumentando varianza. $k=5$ y $k=10$ son compromisos estandar.

**Demostracion algebraica del sesgo.** Sea $f^{(-i)}$ entrenado con $n(1-1/k)$ datos. El error asintotico del aprendizaje clasico se expande como $R(f^{(-i)}) = R(f^{*}) + O(1/(n(1-1/k)))$. Promediando, $\mathbb{E}[\hat R_{CV}] = R(f^{*}) + O(1/n)$ cuando $k$ es fijo. Por lo tanto CV **reduce el sesgo** respecto a un solo holdout (que usa $n(1-\alpha)$ con $\alpha=0.25$ tipicamente) porque promedia sobre $k$ evaluaciones, no una.

</font>


In [ ]:
resultados_split = []
for prop in [0.70, 0.75, 0.80]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, train_size=prop,
                                          stratify=y, random_state=RANDOM_STATE)
    m = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    sc = StandardScaler()
    m.fit(sc.fit_transform(Xtr), ytr)
    auc = roc_auc_score(yte, m.predict_proba(sc.transform(Xte))[:,1])
    resultados_split.append({'prop_train': prop, 'n_train': len(Xtr),
                             'n_test': len(Xte), 'AUC_test': auc})
res_split = pd.DataFrame(resultados_split)
print('Varianza AUC entre splits:', res_split['AUC_test'].var())
display(res_split.round(4))

<font face="Times New Roman">

Se compararon tres proporciones de train/test (70/30, 75/25, 80/20). Los AUCs salen 0.6587, 0.6581 y 0.6588 — varianza ≈ 1.8e-7, diferencias en el cuarto decimal. Usamos 75/25 como compromiso estándar.

</font>


<font face="Times New Roman">

Aquí se compara el efecto de tres estrategias de balanceo en una regresión logística simple. En **AUC** las tres dan ≈0.658, idénticas. En **F1-macro** sí hay diferencia: sin balanceo 0.554, `class_weight='balanced'` 0.606, SMOTE 0.606. El balanceo no cambia el ranking de las observaciones, solo el corte óptimo a threshold 0.5.

</font>


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, train_size=0.75, stratify=y, random_state=RANDOM_STATE)
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

def eval_auc_f1(clf, Xtr, ytr, Xte, yte):
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:,1]
    return (roc_auc_score(yte, p),
            f1_score(yte, (p>0.5).astype(int), average='macro'))

rows = []
rows.append(('sin balanceo', *eval_auc_f1(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), Xtr_s, ytr, Xte_s, yte)))
rows.append(("class_weight='balanced'", *eval_auc_f1(
    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    Xtr_s, ytr, Xte_s, yte)))
if HAS_IMBLEARN:
    smo = SMOTE(random_state=RANDOM_STATE)
    Xb, yb = smo.fit_resample(Xtr_s, ytr)
    rows.append(('SMOTE', *eval_auc_f1(
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), Xb, yb, Xte_s, yte)))
display(pd.DataFrame(rows, columns=['estrategia','AUC','F1-macro']).round(4))

<font face="Times New Roman">

CV estratificada con $k=5$ y $k=10$ folds. Resultados: $k=5$ AUC = **0.6569 ± 0.0026**, $k=10$ AUC = **0.6570 ± 0.0032**. La media casi idéntica y la desviación pequeña confirman que la estimación es estable. Usaremos $k=5$ por defecto en el resto del notebook por costo computacional.

</font>


In [ ]:
pipe = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
for k in [5, 10]:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    aucs = cross_val_score(pipe, StandardScaler().fit_transform(X), y, cv=skf, scoring='roc_auc')
    print(f'k={k:2d}  AUC = {aucs.mean():.4f} +- {aucs.std():.4f}')

<font face="Times New Roman">

## Parte 4 - Modelos `[50 pts = 10 x 5]`

Construimos una funcion que evalua el producto cartesiano de configuraciones (hiperparametros x proporciones x balanceo x transformaciones), calcula la metrica compuesta

$$\lambda_{\text{score}} = (1-\lambda)\,\text{AUC}_{CV} - \lambda\,|\text{AUC}_{\text{train}} - \text{AUC}_{CV}|,$$

y marca el ganador ($\lambda = 0.5$ por defecto para balancear desempeno y gap de overfitting).

</font>


In [ ]:
from itertools import product
from time import perf_counter

def pipeline_experimentos(estimator_factory, param_grid, X, y,
                          props=(0.70, 0.75, 0.80),
                          balanceo=('none','class_weight','smote'),
                          estandarizar=True, k_cv=5, lam=0.5,
                          max_configs=None, verbose=False):
    """Evalua producto cartesiano y devuelve DataFrame con ganador."""
    keys = list(param_grid.keys())
    grids = list(product(*[param_grid[k] for k in keys]))
    combos = list(product(grids, props, balanceo))
    if max_configs: combos = combos[:max_configs]
    rows = []
    skf = StratifiedKFold(n_splits=k_cv, shuffle=True, random_state=RANDOM_STATE)

    for (gridvals, prop, bal) in combos:
        params = dict(zip(keys, gridvals))
        Xtr, Xte, ytr, yte = train_test_split(X, y, train_size=prop,
                                              stratify=y, random_state=RANDOM_STATE)
        if estandarizar:
            sc = StandardScaler().fit(Xtr)
            Xtr_ = sc.transform(Xtr); Xte_ = sc.transform(Xte)
        else:
            Xtr_ = Xtr.values if hasattr(Xtr,'values') else Xtr
            Xte_ = Xte.values if hasattr(Xte,'values') else Xte

        params_fit = dict(params)
        if bal == 'class_weight' and 'class_weight' in estimator_factory().get_params():
            params_fit['class_weight'] = 'balanced'

        if bal == 'smote' and HAS_IMBLEARN:
            Xtr_bal, ytr_bal = SMOTE(random_state=RANDOM_STATE).fit_resample(Xtr_, ytr)
        else:
            Xtr_bal, ytr_bal = Xtr_, ytr

        t0 = perf_counter()
        try:
            clf = estimator_factory(**{k:v for k,v in params_fit.items()
                                        if k in estimator_factory().get_params()})
            cv_scores = []
            for tr_idx, va_idx in skf.split(Xtr_, ytr):
                Xa, Xb = Xtr_[tr_idx], Xtr_[va_idx]
                ya, yb = ytr.iloc[tr_idx].values, ytr.iloc[va_idx].values
                if bal == 'smote' and HAS_IMBLEARN:
                    Xa, ya = SMOTE(random_state=RANDOM_STATE).fit_resample(Xa, ya)
                clf.fit(Xa, ya)
                cv_scores.append(roc_auc_score(yb, clf.predict_proba(Xb)[:,1]))
            cv_mean, cv_std = np.mean(cv_scores), np.std(cv_scores)
            clf.fit(Xtr_bal, ytr_bal)
            auc_tr = roc_auc_score(ytr, clf.predict_proba(Xtr_)[:,1])
            p_te   = clf.predict_proba(Xte_)[:,1]
            auc_te = roc_auc_score(yte, p_te)
            f1_te  = f1_score(yte, (p_te>0.5).astype(int), average='macro')
            t = perf_counter() - t0
            lam_score = (1-lam)*cv_mean - lam*abs(auc_tr - cv_mean)
            rows.append({**params, 'prop_train':prop, 'balanceo':bal,
                         'AUC_train':auc_tr, 'AUC_CV_mean':cv_mean, 'AUC_CV_std':cv_std,
                         'AUC_test':auc_te, 'F1_test':f1_te, 't_s':t,
                         'lambda_score':lam_score})
            if verbose: print(rows[-1])
        except Exception as e:
            if verbose: print('FAIL', params, prop, bal, e)
    res = pd.DataFrame(rows).sort_values('lambda_score', ascending=False).reset_index(drop=True)
    res['Ganador'] = ''
    if len(res): res.loc[0, 'Ganador'] = 'X'
    return res

<font face="Times New Roman">

La función `pipeline_experimentos` sirve para la evaluación sistemática de modelos. Explora un producto cartesiano de hiperparámetros, proporciones de split y estrategias de balanceo. Calcula métricas en train, CV y test, y un $\lambda$-score ponderado que penaliza el overfitting, seleccionando automáticamente la mejor configuración.

</font>


<font face="Times New Roman">

### 4.1 Regresion Logistica / MLG Bernoulli

**Bernoulli en la familia exponencial.** La familia Bernoulli con probabilidad $p$ tiene densidad

$$f(y; p) = p^y (1-p)^{1-y} = \exp\left\{ y\log\frac{p}{1-p} + \log(1-p) \right\}.$$

Comparando con la forma canonica $f(y;\theta) = \exp\{y\theta - b(\theta) + c(y,\phi)\}$, identificamos el **parametro natural** $\theta = \log\frac{p}{1-p}$ (logit), $b(\theta) = \log(1+e^\theta)$, $\phi=1$, $c(y,\phi)=0$. Por lo tanto el **enlace canonico** es $g(p) = \log\frac{p}{1-p}$, justificando el uso de la regresion logistica como MLG optima en terminos de suficiencia.

</font>


<font face="Times New Roman">

### Implementación y Evaluación de Regresión Logística

Para la regresión logística aplicamos `pipeline_experimentos` explorando `C`, proporciones de split y estrategias de balanceo. La estandarización es necesaria para que la regularización L2 trate todas las variables en pie de igualdad. El ganador se selecciona maximizando $\lambda$-score = $0.5 \cdot \text{AUC}_{CV} - 0.5 \cdot |\text{AUC}_{train} - \text{AUC}_{CV}|$.

</font>


In [ ]:
grid_lr = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
}
def fab_lr(**kw):
    base = dict(max_iter=3000, random_state=RANDOM_STATE)
    base.update(kw); return LogisticRegression(**base)

# Subsample data for faster tuning, as done for SVM/MLP
np.random.seed(RANDOM_STATE)
idx_lr = np.random.choice(len(X), size=min(40_000, len(X)), replace=False)
X_lr_subsample = X.iloc[idx_lr].reset_index(drop=True)
y_lr_subsample = y.iloc[idx_lr].reset_index(drop=True)

res_lr = pipeline_experimentos(fab_lr, grid_lr, X_lr_subsample, y_lr_subsample,
                               props=(0.75, 0.80),
                               balanceo=('class_weight','smote') if HAS_IMBLEARN else ('class_weight',),
                               estandarizar=True, k_cv=5, lam=0.5)
print(f'Configuraciones evaluadas: {len(res_lr)}')
display(res_lr.round(4).head(20))

<font face="Times New Roman">

Se ejecuta `pipeline_experimentos` para Regresión Logística sobre un submuestreo de **40,000 observaciones** (para acelerar). Se exploran `C` ∈ {0.01, 0.1, 1, 10, 100}, dos proporciones (0.75 y 0.80) y dos estrategias de balanceo (`class_weight` y SMOTE). El **ganador** es `C=0.10, class_weight=balanced, prop_train=0.80` con **AUC-CV = 0.6494 ± 0.0056**, AUC-test = **0.6533**, F1-test = **0.6040** y $\lambda$-score = **0.3242**.

</font>


In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

Xc = StandardScaler().fit_transform(X)
model = sm.Logit(y, sm.add_constant(Xc)).fit(disp=0)
coef = model.params[1:]; ci = model.conf_int()[1:]
or_df = pd.DataFrame({'feature': X.columns,
                      'coef': coef.values,
                      'OR': np.exp(coef.values),
                      'OR_lo': np.exp(ci[0].values),
                      'OR_hi': np.exp(ci[1].values)}).sort_values('OR', ascending=False)
display(or_df.round(3))

<font face="Times New Roman">

Aquí calculamos los Odds Ratios con `statsmodels`. Como `X` está estandarizado, cada OR mide el cambio multiplicativo en las odds de despacho ante un cambio de **una desviación estándar** en la feature. Los efectos positivos más fuertes son **tipo_entrada_woe** (OR=1.578), **duracion_min** (1.533), **incidente_c4_woe** (1.295) y **alcaldia_hechos_woe** (1.148). Los efectos protectores más fuertes son **hora_creacion_num** (OR=0.726) y **hora_cos** (0.784): incidentes en la primera mitad del día tienen menor probabilidad de despacho una vez controladas las demás variables.

</font>


<font face="Times New Roman">

Esta celda replica el análisis de Odds Ratios en R, utilizando la función `glm` para ajustar un modelo de regresión logística. También se muestran los coeficientes del modelo y sus respectivos Odds Ratios con intervalos de confianza al 95%.

</font>


In [ ]:
%%R -i X -i y
X_df <- as.data.frame(X)
X_df$y <- as.factor(y)
fit <- glm(y ~ ., data=X_df, family=binomial(link='logit'))
cat('=== coef summary ===\n')
print(summary(fit)$coefficients)
cat('\n=== odds ratios + IC 95% ===\n')
print(exp(cbind(OR=coef(fit), confint.default(fit))))

<font face="Times New Roman">

### 4.2 Arbol de decision

La impureza de Gini de un nodo con proporcion de positivos $p$ es $G(p) = 2p(1-p)$. La entropia es $H(p) = -p\log p - (1-p)\log(1-p)$. Ambas son concavas, simetricas en $p=0.5$, y tienen el mismo orden de magnitud en $[0,1]$. El split $s$ maximiza la reduccion de impureza $\Delta I = I(\text{padre}) - \frac{n_L}{n}I(L) - \frac{n_R}{n}I(R)$. Para Gini, minimizar la impureza ponderada equivale a maximizar la ganancia de informacion bajo una aproximacion de primer orden (desarrollo de Taylor de $\log$ alrededor de $p=0.5$), y en la practica ambos criterios producen arboles casi identicos en clasificacion binaria balanceada.

</font>


In [ ]:
grid_tree = {
    'criterion': ['gini','entropy'],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [20, 100],
    'ccp_alpha': [0.0, 0.001],
}
def fab_tree(**kw):
    base = dict(random_state=RANDOM_STATE)
    base.update(kw); return DecisionTreeClassifier(**base)

np.random.seed(RANDOM_STATE)
idx_tree = np.random.choice(len(X), size=min(40_000, len(X)), replace=False)
X_tree_subsample = X.iloc[idx_tree].reset_index(drop=True)
y_tree_subsample = y.iloc[idx_tree].reset_index(drop=True)

res_tree = pipeline_experimentos(fab_tree, grid_tree, X_tree_subsample, y_tree_subsample,
                                 props=(0.75,),
                                 balanceo=('class_weight','smote') if HAS_IMBLEARN else ('class_weight',),
                                 estandarizar=False, k_cv=5, lam=0.5,
                                 max_configs=40)
print(f'Configuraciones evaluadas: {len(res_tree)}')
display(res_tree.round(4).head(15))

<font face="Times New Roman">

Esta celda define la grilla del árbol: criterios `gini`/`entropy`, `max_depth` ∈ {3, 5, 7, 10}, `min_samples_split` ∈ {20, 100} y `ccp_alpha` ∈ {0, 0.001}. Se evalúan **40 configuraciones**. El **ganador** es `criterion=gini, max_depth=10, min_samples_split=20, ccp_alpha=0.001, balanceo=smote` con AUC-CV = **0.6433 ± 0.0068**, AUC-test = **0.6489**, F1-test = **0.5589** y $\lambda$-score = **0.3182**.

</font>


In [ ]:
best = res_tree.iloc[0]
params_best = {k: best[k] for k in ['criterion','max_depth','min_samples_split','ccp_alpha']}
tree_best = DecisionTreeClassifier(**params_best, class_weight='balanced', random_state=RANDOM_STATE)
Xtr, Xte, ytr, yte = train_test_split(X, y, train_size=0.75, stratify=y, random_state=RANDOM_STATE)
tree_best.fit(Xtr, ytr)

fig, ax = plt.subplots(figsize=(14,7))
plot_tree(tree_best, feature_names=X.columns, class_names=['no','despacho'],
          filled=True, rounded=True, max_depth=3, fontsize=9, ax=ax)
plt.title(f'Arbol ganador (depth={params_best["max_depth"]}, crit={params_best["criterion"]})')
plt.tight_layout(); plt.show()

imp = pd.Series(tree_best.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(7,5))
imp.plot.barh(ax=ax, color='#3d5a80')
ax.set_title('Importancia de variables - arbol')
plt.tight_layout(); plt.show()

<font face="Times New Roman">

Reentrenamos el árbol ganador (gini, depth=10, ccp_α=0.001) sobre el dataset completo y lo visualizamos. La importancia se concentra en `tipo_entrada_woe` y `duracion_min`, coherente con los OR de la regresión.

</font>


<font face="Times New Roman">

Implementación equivalente en R con `rpart`. Con `cp=0.001, minsplit=50, maxdepth=5` el árbol usa cinco variables: `duracion_min`, `hora_creacion_num`, `hora_sin`, `incidente_c4_woe` y `tipo_entrada_woe`.

</font>


In [ ]:
%%R -i X -i y
if (!requireNamespace("rpart.plot", quietly = TRUE)) {
    install.packages("rpart.plot", repos = "http://cran.us.r-project.org")
}
library(rpart); library(rpart.plot)
d <- as.data.frame(X); d$y <- as.factor(y)
ctrl <- rpart.control(cp=0.001, minsplit=50, maxdepth=5)
fit <- rpart(y ~ ., data=d, method='class', control=ctrl)
cat('=== printcp ===\n'); printcp(fit)
cat('\n=== podado al cp optimo ===\n')
cp_opt <- fit$cptable[which.min(fit$cptable[,'xerror']),'CP']
fit_p  <- prune(fit, cp=cp_opt)
print(fit_p$variable.importance)

<font face="Times New Roman">

### 4.3 SVM

El primal SVM con margen blando es

$$\min_{w,b,\xi} \tfrac{1}{2}\|w\|^2 + C\sum_i \xi_i \quad \text{s.a. } y_i(w^\top x_i + b) \ge 1-\xi_i,\ \xi_i \ge 0.$$

Con multiplicadores $\alpha_i$ se llega al problema dual

$$\max_\alpha \sum_i \alpha_i - \tfrac{1}{2}\sum_{i,j}\alpha_i\alpha_j y_i y_j \,\langle x_i, x_j\rangle \quad \text{s.a. } 0\le\alpha_i\le C,\ \sum_i \alpha_i y_i = 0.$$

En el dual solo aparece el producto interno $\langle x_i, x_j\rangle$, que puede reemplazarse por un kernel $K(x,x')$ (*kernel trick*)

</font>


In [ ]:
import numpy as np # Ensure numpy is imported

np.random.seed(RANDOM_STATE)
idx = np.random.choice(len(X), size=min(5_000, len(X)), replace=False) # Further reduced subsample size for faster execution
X_svm = X.iloc[idx].reset_index(drop=True)
y_svm = y.iloc[idx].reset_index(drop=True);

grid_svm = {
    'kernel': ['linear','rbf'],
    'C': [1.0, 10.0],
    'gamma': ['scale', 0.1],
}
def fab_svm(**kw):
    base = dict(probability=True, random_state=RANDOM_STATE, cache_size=500)
    base.update(kw); return SVC(**base)

res_svm = pipeline_experimentos(fab_svm, grid_svm, X_svm, y_svm,
                                props=(0.75,),
                                balanceo=('class_weight',),
                                estandarizar=True, k_cv=3, lam=0.5,
                                max_configs=6)
print(f'Configuraciones SVM: {len(res_svm)}')
display(res_svm.round(4).head(15))

<font face="Times New Roman">

Ejecuta `pipeline_experimentos` para SVM. Por la complejidad $O(n^2)$ se trabaja con un submuestreo de **5,000 observaciones** y CV de 3 folds. Se evalúan kernels `linear` y `rbf`, `C` ∈ {1, 10}, `gamma` ∈ {`scale`, 0.1}. El **ganador** es `kernel=linear, C=10, balanceo=class_weight` con AUC-CV = **0.6457 ± 0.0194**, AUC-test = **0.6255**, F1-test = **0.5349** y $\lambda$-score = **0.3166**. Las configuraciones RBF muestran AUC-train de 0.74–0.77 vs AUC-CV de 0.63 — el $\lambda$-score las penaliza por overfitting.

</font>


<font face="Times New Roman">

Para visualizar la frontera del SVM ganador, proyectamos a 2D con PCA y reentrenamos.

</font>


In [ ]:
# Frontera de decision via PCA a 2D
best_svm_params = res_svm.iloc[0][['kernel','C','gamma']].to_dict()
sc = StandardScaler().fit(X_svm)
Xs = sc.transform(X_svm)
pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Xs)
X2 = pca.transform(Xs)

svm2 = SVC(**best_svm_params, probability=False, random_state=RANDOM_STATE, class_weight='balanced')
svm2.fit(X2, y_svm)

xx, yy = np.meshgrid(np.linspace(X2[:,0].min()-.5, X2[:,0].max()+.5, 200),
                     np.linspace(X2[:,1].min()-.5, X2[:,1].max()+.5, 200))
Z = svm2.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7,5.5))
ax.contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.5)
ax.scatter(X2[::10,0], X2[::10,1], c=y_svm[::10], cmap='RdBu_r',
           edgecolor='k', s=12, linewidths=0.3)
ax.set_title(f'Frontera SVM ({best_svm_params["kernel"]}) en PCA 2D')
plt.tight_layout(); plt.show()
print(f'#SV = {svm2.support_.size} ({svm2.support_.size/len(X2)*100:.1f}% del dataset)')

<font face="Times New Roman">

Implementación de SVM en R con `e1071`. Sobre un submuestreo de 4,000 observaciones, kernel radial, cost=1, gamma=0.1: el modelo reporta **#SV ≈ 2,978 (≈74%)**, consistente con el observado en Python para configuración RBF.

</font>


In [ ]:
%%R -i X_svm -i y_svm
if (!requireNamespace("e1071", quietly = TRUE)) {
    install.packages("e1071", repos = "http://cran.us.r-project.org")
}
library(e1071)
d <- as.data.frame(X_svm); d$y <- as.factor(y_svm)
set.seed(42)
idx <- sample(seq_len(nrow(d)), 4000)
fit <- svm(y ~ ., data=d[idx,], kernel='radial', cost=1, gamma=0.1,
           probability=TRUE, scale=TRUE)
cat('#SV:', fit$tot.nSV, '\n')
cat('Summary:\n'); print(summary(fit))

<font face="Times New Roman">

### 4.4 K-Vecinos más Cercanos

**Cota de Cover-Hart.** Para $n \to \infty$ y $k=1$, el error asintótico del 1-NN satisface

$$R^{*} \le R_{1NN} \le 2 R^{*},$$

donde $R^{*}$ es el error de Bayes. Es decir, incluso 1-NN no puede ser más del doble de malo que el clasificador óptimo asintóticamente — una garantía teórica notable para un método sin parámetros.

**KNN desde cero (+5 pts).** Implementamos `KNNFromScratch` con distancias vectorizadas en `numpy`. La validación contra `sklearn.KNeighborsClassifier` da 100% de acuerdo en una muestra de 200 observaciones.

</font>


<font face="Times New Roman">

La clase `KNNFromScratch` es una implementación desde cero del algoritmo K-Nearest Neighbors para clasificación. Permite especificar el número de vecinos `k`, la métrica de distancia (`euclidean`, `manhattan`, `minkowski`), y el método de ponderación (`uniform` o `distance`). La verificación final compara su `predict` con el `KNeighborsClassifier` de scikit-learn en un pequeño subconjunto de datos para asegurar su correcto funcionamiento.

</font>


In [ ]:
class KNNFromScratch:
    """KNN desde cero. Reproduce sklearn.KNeighborsClassifier para distancias
    euclidiana/manhattan/minkowski con pesos uniformes o por distancia inversa."""
    def __init__(self, k=5, metric='euclidean', p=2, weights='uniform'):
        self.k = k; self.metric = metric; self.p = p; self.weights = weights

    def fit(self, X, y):
        self.X_ = np.asarray(X, dtype=float)
        self.y_ = np.asarray(y)
        self.classes_ = np.unique(self.y_)
        return self

    def _dist(self, Xq):
        if self.metric == 'euclidean':
            d = np.sqrt(((Xq[:,None,:] - self.X_[None,:,:])**2).sum(-1))
        elif self.metric == 'manhattan':
            d = np.abs(Xq[:,None,:] - self.X_[None,:,:]).sum(-1)
        elif self.metric == 'minkowski':
            d = (np.abs(Xq[:,None,:] - self.X_[None,:,:])**self.p).sum(-1)**(1/self.p)
        else:
            raise ValueError(self.metric)
        return d

    def predict_proba(self, Xq):
        Xq = np.asarray(Xq, dtype=float)
        out = np.zeros((len(Xq), len(self.classes_)))
        bloque = 500
        for i in range(0, len(Xq), bloque):
            D = self._dist(Xq[i:i+bloque])
            idx_k = np.argpartition(D, self.k, axis=1)[:, :self.k]
            for j in range(len(idx_k)):
                etqs = self.y_[idx_k[j]]
                if self.weights == 'uniform':
                    w = np.ones(self.k)
                else:
                    dists = D[j, idx_k[j]]
                    w = 1.0 / (dists + 1e-9)
                for c_idx, c in enumerate(self.classes_):
                    out[i+j, c_idx] = w[etqs == c].sum()
                out[i+j] /= out[i+j].sum()
        return out

    def predict(self, Xq):
        return self.classes_[self.predict_proba(Xq).argmax(1)]

# Validacion: comparar contra sklearn en muestra pequena
from sklearn.neighbors import KNeighborsClassifier as SKKNN
Xv = StandardScaler().fit_transform(X.iloc[:2000])
yv = y.iloc[:2000].values
Xq = Xv[-200:]
k1 = KNNFromScratch(k=7, metric='euclidean').fit(Xv[:-200], yv[:-200])
k2 = SKKNN(n_neighbors=7, metric='euclidean').fit(Xv[:-200], yv[:-200])
acc_match = (k1.predict(Xq) == k2.predict(Xq)).mean()
print(f'Acuerdo KNNFromScratch vs sklearn: {acc_match*100:.1f}%')

<font face="Times New Roman">

Tabla de hiperparámetros KNN: `n_neighbors` ∈ {3, 5, 7, 11, 15, 21, 31}, `metric` ∈ {`euclidean`, `manhattan`}, `weights` ∈ {`uniform`, `distance`}. Sobre 20,000 observaciones y 3 folds se evalúan 28 configuraciones. El **ganador** es `n_neighbors=31, metric=euclidean, weights=uniform` con AUC-CV = **0.6205 ± 0.0123**, AUC-test = **0.6162**, F1-test = **0.5452** y $\lambda$-score = **0.2747**. Las configuraciones con `weights=distance` colapsan a AUC-train = 1.0 (memorización pura) y son penalizadas correctamente.

</font>


In [ ]:
grid_knn = {
    'n_neighbors': [3, 5, 7, 11, 15, 21, 31],
    'metric': ['euclidean','manhattan'],
    'weights': ['uniform','distance'],
}
def fab_knn(**kw):
    return KNeighborsClassifier(**kw, n_jobs=-1)

idx = np.random.RandomState(42).choice(len(X), size=20_000, replace=False)
res_knn = pipeline_experimentos(fab_knn, grid_knn, X.iloc[idx], y.iloc[idx],
                                props=(0.75,), balanceo=('none',),
                                estandarizar=True, k_cv=3, lam=0.5,
                                max_configs=30)
print(f'Configuraciones KNN: {len(res_knn)}')
display(res_knn.round(4).head(15))

<font face="Times New Roman">

Curva AUC-CV vs $k$. La AUC sube **monótonamente** de ≈0.55 (k=1) a ≈0.63 (k=31): valores pequeños de $k$ sobreajustan al ruido local y valores grandes promedian mejor. El óptimo está en el extremo superior del rango.

</font>


In [ ]:
ks = [1, 3, 5, 7, 11, 15, 21, 31]
n_sub = 20_000
Xs_full = StandardScaler().fit_transform(X.iloc[:n_sub].values)
ys_full = y.iloc[:n_sub].reset_index(drop=True).values

assert not np.isnan(Xs_full).any(), "X tiene NaN antes de KNN"
assert not np.isinf(Xs_full).any(), "X tiene Inf antes de KNN"

skf_k = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
filas = []
for kk in ks:
    aucs = []
    for tr_idx, va_idx in skf_k.split(Xs_full, ys_full):
        clf = KNeighborsClassifier(n_neighbors=kk, n_jobs=1)
        clf.fit(Xs_full[tr_idx], ys_full[tr_idx])
        p = clf.predict_proba(Xs_full[va_idx])[:, 1]
        aucs.append(roc_auc_score(ys_full[va_idx], p))
    filas.append((kk, float(np.mean(aucs)), float(np.std(aucs))))
    print(f"  k={kk:2d}  AUC = {filas[-1][1]:.4f} +- {filas[-1][2]:.4f}")

df_k = pd.DataFrame(filas, columns=["k", "AUC_CV", "AUC_std"])
k_opt = int(df_k.loc[df_k["AUC_CV"].idxmax(), "k"])

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.errorbar(df_k["k"], df_k["AUC_CV"], yerr=df_k["AUC_std"],
            fmt="o-", color="#3d5a80", lw=2, capsize=4, label="AUC CV (3 folds)")
ax.axvline(k_opt, color="#c8553d", ls="--", lw=1, label=f"k optimo = {k_opt}")
ax.set_xlabel("k (numero de vecinos)")
ax.set_ylabel("AUC 3-fold CV")
ax.set_title("Curva AUC-CV vs k - KNN")
ax.set_xticks(ks)
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()



<font face="Times New Roman">

Implementación equivalente en R con el paquete `class`. Con `k=7` sobre 2,000 observaciones de prueba: accuracy = **0.6455**, consistente con la tasa de no-despacho (63.5%) más algo de información extra de los WoE.

</font>


In [ ]:
%%R -i X -i y
library(class)
Xs <- scale(as.matrix(X))
set.seed(42)
idx <- sample(seq_len(nrow(Xs)), 8000)
tr <- sample(idx, 6000); te <- setdiff(idx, tr)
pred <- knn(train=Xs[tr,], test=Xs[te,], cl=y[tr], k=7, prob=TRUE)
cat('Accuracy KNN k=7:', mean(pred == y[te]), '\n')

<font face="Times New Roman">

### 4.5 Red Neuronal — MLP Shallow

**Teorema de aproximación universal (Cybenko 1989, Hornik 1991).** Sea $\sigma$ una activación no constante, acotada y monótona creciente (o no polinómica como ReLU). El conjunto de funciones

$$\mathcal{F}_h = \left\{ f(x) = \sum_{j=1}^{h} \alpha_j \sigma(w_j^\top x + b_j) \right\}$$

es **denso** en $C(K)$ (continuas sobre compactos) en norma uniforme. Es decir, un MLP con **una sola capa oculta** y suficientes neuronas aproxima cualquier función continua.

**Backpropagation — esquema una capa oculta.** Sea $z^{(1)} = W^{(1)}x + b^{(1)}$, $a^{(1)} = \sigma(z^{(1)})$, $\hat y = \text{softmax}(W^{(2)}a^{(1)} + b^{(2)})$, $\mathcal L = -\sum_c y_c \log \hat y_c$. La regla de la cadena da:

$$\frac{\partial \mathcal L}{\partial W^{(1)}} = \big[ (W^{(2)})^\top(\hat y - y) \odot \sigma'(z^{(1)}) \big] \cdot x^\top.$$

Ese gradiente se calcula eficientemente con backprop: forward pass primero, luego propagación hacia atrás del error $\delta$.

</font>


In [ ]:
p = X.shape[1]
grid_mlp = {
    'hidden_layer_sizes': [(p,), (int(1.5*p),), (2*p,), (p, p//2)],
    'activation': ['relu','tanh'],
    'alpha': [1e-4, 1e-3, 1e-2],
    'learning_rate_init': [1e-3],
}
def fab_mlp(**kw):
    base = dict(max_iter=300, early_stopping=True, validation_fraction=0.1,
                random_state=RANDOM_STATE)
    base.update(kw); return MLPClassifier(**base)

idx = np.random.RandomState(42).choice(len(X), size=40_000, replace=False)
res_mlp = pipeline_experimentos(fab_mlp, grid_mlp, X.iloc[idx], y.iloc[idx],
                                props=(0.75,), balanceo=('none',),
                                estandarizar=True, k_cv=3, lam=0.5,
                                max_configs=24)
print(f'Configuraciones MLP: {len(res_mlp)}')
display(res_mlp.round(4).head(15))

In [ ]:
best = res_mlp.iloc[0]
mlp_best = MLPClassifier(
    hidden_layer_sizes=best['hidden_layer_sizes'],
    activation=best['activation'],
    alpha=best['alpha'],
    learning_rate_init=best['learning_rate_init'],
    max_iter=300, random_state=RANDOM_STATE
)
Xs = StandardScaler().fit_transform(X.iloc[idx])
mlp_best.fit(Xs, y.iloc[idx])

fig, ax = plt.subplots(figsize=(6,3.5))
ax.plot(mlp_best.loss_curve_, color='#ee6c4d')
ax.set_xlabel('Iteracion'); ax.set_ylabel('Loss')
ax.set_title(f'Curva de perdida MLP (iters={len(mlp_best.loss_curve_)})')
plt.tight_layout(); plt.show()
if len(mlp_best.loss_curve_) >= 300 - 1:
    print('ADVERTENCIA: MLP no convergio en max_iter=300; considerar aumentar iters o alpha.')
else:
    print('OK: MLP convergio.')

In [ ]:
# EXTRA: Heatmap pesos primera capa
W1 = mlp_best.coefs_[0]   # shape (p, h1)
fig, ax = plt.subplots(figsize=(9, max(3, 0.35*len(X.columns))))
sns.heatmap(W1, cmap='RdBu_r', center=0, yticklabels=list(X.columns),
            xticklabels=[f'h{i}' for i in range(W1.shape[1])], ax=ax)
ax.set_title('Pesos MLP capa 1 - "importancia" visual por feature -> neurona oculta')
plt.tight_layout(); plt.show()

In [ ]:
%%R -i X -i y
library(nnet)
Xs <- scale(as.matrix(X))
set.seed(42)
idx <- sample(seq_len(nrow(Xs)), 6000)
fit <- nnet(x=Xs[idx,], y=class.ind(factor(y[idx])), size=10,
            softmax=TRUE, maxit=200, trace=FALSE)
cat('Final value:', fit$value, '\n')
cat('Converged:', fit$convergence, '\n')

<font face="Times New Roman">

## Parte 5 — Comparación y Conclusiones ``

En esta sección consolidamos los cinco modelos ganadores y los evaluamos en el mismo conjunto de test (49,999 observaciones). Reportamos AUC, F1, pérdida esperada bajo costo asimétrico, calibración (Brier) y costo computacional, además del test de DeLong para verificar si las diferencias entre AUCs son estadísticamente significativas.

</font>


In [ ]:
def fila_ganador(res, nombre):
    if len(res)==0: return None
    r = res.iloc[0]
    return {'Metodo': nombre,
            'AUC_train': round(r['AUC_train'],4),
            'AUC_CV':    f"{r['AUC_CV_mean']:.4f} +- {r['AUC_CV_std']:.4f}",
            'AUC_test':  round(r['AUC_test'],4),
            'F1_test':   round(r['F1_test'],4),
            'tiempo_s':  round(r['t_s'],2),
            'lambda_score': round(r['lambda_score'],4)}

comparativa = pd.DataFrame([
    fila_ganador(res_lr,  'Reg. Logistica'),
    fila_ganador(res_tree,'Arbol'),
    fila_ganador(res_svm, 'SVM'),
    fila_ganador(res_knn, 'KNN'),
    fila_ganador(res_mlp, 'MLP'),
])
display(comparativa)

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, train_size=0.75, stratify=y, random_state=RANDOM_STATE)
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

ganadores = {}
ganadores['Reg.Log.'] = LogisticRegression(
    C=float(res_lr.iloc[0]['C']), max_iter=3000,
    class_weight='balanced', random_state=RANDOM_STATE).fit(Xtr_s, ytr)
ganadores['Arbol'] = DecisionTreeClassifier(
    criterion=res_tree.iloc[0]['criterion'],
    max_depth=int(res_tree.iloc[0]['max_depth']),
    min_samples_split=int(res_tree.iloc[0]['min_samples_split']),
    ccp_alpha=float(res_tree.iloc[0]['ccp_alpha']),
    class_weight='balanced', random_state=RANDOM_STATE).fit(Xtr, ytr)
ganadores['KNN'] = KNeighborsClassifier(
    n_neighbors=int(res_knn.iloc[0]['n_neighbors']),
    metric=res_knn.iloc[0]['metric'],
    weights=res_knn.iloc[0]['weights'], n_jobs=-1).fit(Xtr_s, ytr)
ganadores['MLP'] = MLPClassifier(
    hidden_layer_sizes=res_mlp.iloc[0]['hidden_layer_sizes'],
    activation=res_mlp.iloc[0]['activation'],
    alpha=float(res_mlp.iloc[0]['alpha']),
    max_iter=300, random_state=RANDOM_STATE).fit(Xtr_s, ytr)
svm_idx = np.random.RandomState(42).choice(len(Xtr), size=5_000, replace=False)
ganadores['SVM'] = SVC(
    kernel=res_svm.iloc[0]['kernel'],
    C=float(res_svm.iloc[0]['C']),
    gamma=res_svm.iloc[0]['gamma'],
    probability=True, class_weight='balanced',
    random_state=RANDOM_STATE, cache_size=500
).fit(Xtr_s[svm_idx], ytr.iloc[svm_idx])

probs_te = {n: g.predict_proba(Xte_s if n!='Arbol' else Xte)[:,1] for n, g in ganadores.items()}
print('Ganadores reentrenados.')

In [ ]:
# Curvas ROC + punto de Youden
fig, ax = plt.subplots(figsize=(7,6))
thresholds_youden = {}
for n, p in probs_te.items():
    fpr, tpr, thr = roc_curve(yte, p)
    auc = roc_auc_score(yte, p)
    J = tpr - fpr
    idx_j = np.argmax(J)
    thresholds_youden[n] = thr[idx_j]
    ax.plot(fpr, tpr, lw=2, label=f'{n} (AUC={auc:.3f}, t*={thr[idx_j]:.2f})')
    ax.plot(fpr[idx_j], tpr[idx_j], 'o', ms=7)
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('Curvas ROC + punto de Youden')
ax.legend()
plt.tight_layout(); plt.show()
display(pd.DataFrame({'Metodo': thresholds_youden.keys(),
                      'threshold_Youden': [round(t,3) for t in thresholds_youden.values()]}))

In [ ]:
# Matriz de costos actuarial
C_FN, C_FP = 5000, 1000

def perdida_esperada(y_true, p_hat, thr, c_fn=C_FN, c_fp=C_FP):
    yhat = (p_hat > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yhat).ravel()
    return (fn*c_fn + fp*c_fp) / len(y_true)

perdidas = {n: perdida_esperada(yte, probs_te[n], thresholds_youden[n])
            for n in probs_te}
display(pd.DataFrame({'Metodo':perdidas.keys(),
                      'perdida_esperada_$':[round(v,2) for v in perdidas.values()]})
        .sort_values('perdida_esperada_$'))

In [ ]:
# Test de DeLong: diferencia de AUCs pareadas
def delong_pvalue(y_true, p1, p2):
    """Implementacion simplificada de DeLong para dos AUCs correlacionados."""
    y_true = np.asarray(y_true)
    pos = y_true == 1; neg = ~pos
    n1, n0 = pos.sum(), neg.sum()

    def structural(p, y):
        pos_s = p[y==1]; neg_s = p[y==0]
        V10 = np.array([(np.sum(pos_s[i] > neg_s) + 0.5*np.sum(pos_s[i]==neg_s))/n0
                        for i in range(len(pos_s))])
        V01 = np.array([(np.sum(neg_s[j] < pos_s) + 0.5*np.sum(neg_s[j]==pos_s))/n1
                        for j in range(len(neg_s))])
        return V10, V01

    V10_a, V01_a = structural(p1, y_true)
    V10_b, V01_b = structural(p2, y_true)
    auc_a = V10_a.mean(); auc_b = V10_b.mean()
    S10 = np.cov(np.vstack([V10_a, V10_b]))
    S01 = np.cov(np.vstack([V01_a, V01_b]))
    S = S10/n1 + S01/n0
    L = np.array([1, -1])
    var_diff = L @ S @ L
    if var_diff <= 0:
        return auc_a, auc_b, np.nan
    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2*(1 - stats.norm.cdf(abs(z)))
    return auc_a, auc_b, p

pairs = []
names = list(probs_te.keys())
for i, a in enumerate(names):
    for b in names[i+1:]:
        aa, ab, pv = delong_pvalue(yte, probs_te[a], probs_te[b])
        pairs.append({'A':a,'B':b,'AUC_A':round(aa,4),'AUC_B':round(ab,4),
                      'p_value_DeLong': None if np.isnan(pv) else round(pv, 4)})
display(pd.DataFrame(pairs))

<font face="Times New Roman">

### Calibración (+3 pts) y Tiempos (+2 pts)

Para los cinco ganadores ajustamos una calibración isotónica con `FrozenEstimator`, reportamos el Brier score y medimos los tiempos de fit y predict.

</font>


In [ ]:
# Calibracion: isotonic (sklearn 1.8 reemplaza cv="prefit" por FrozenEstimator)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator

fig, ax = plt.subplots(figsize=(7,5))
ax.plot([0,1],[0,1],'k--', lw=1, label='perfectamente calibrado')
brier = {}
for n, g in ganadores.items():
    xx = Xtr_s if n != 'Arbol' else Xtr
    xe = Xte_s if n != 'Arbol' else Xte
    if n == 'SVM':
        cal = CalibratedClassifierCV(estimator=FrozenEstimator(g), method='isotonic')
        cal.fit(xx[svm_idx], ytr.iloc[svm_idx])
    else:
        cal = CalibratedClassifierCV(estimator=FrozenEstimator(g), method='isotonic').fit(xx, ytr)
    p_cal = cal.predict_proba(xe)[:,1]
    frac_pos, mean_pred = calibration_curve(yte, p_cal, n_bins=10)
    brier[n] = brier_score_loss(yte, p_cal)
    ax.plot(mean_pred, frac_pos, 'o-', label=f'{n} (Brier={brier[n]:.4f})')
ax.set_xlabel('Probabilidad predicha (promedio por bin)')
ax.set_ylabel('Fraccion observada de positivos')
ax.set_title('Curvas de calibracion (isotonic)')
ax.legend()
plt.tight_layout(); plt.show()
display(pd.DataFrame({'Metodo': brier.keys(),
                      'Brier_score_isotonic': [round(v,4) for v in brier.values()]})
        .sort_values('Brier_score_isotonic'))


In [ ]:
# Tiempos - train y predict por modelo
tiempos = []
for n, g in ganadores.items():
    xx = Xtr_s if n != 'Arbol' else Xtr
    xe = Xte_s if n != 'Arbol' else Xte
    t0 = time.perf_counter(); g.predict(xe); t_pred = time.perf_counter() - t0
    g2 = g.__class__(**g.get_params())
    try:
        if n == 'SVM':
            t0 = time.perf_counter(); g2.fit(xx[svm_idx], ytr.iloc[svm_idx]); t_fit = time.perf_counter()-t0
        else:
            t0 = time.perf_counter(); g2.fit(xx, ytr); t_fit = time.perf_counter()-t0
    except Exception:
        t_fit = np.nan
    tiempos.append({'Metodo':n,'t_fit_s':round(t_fit,2),'t_predict_s':round(t_pred,3)})
display(pd.DataFrame(tiempos).sort_values('t_fit_s'))

<font face="Times New Roman">

### Conclusiones

**Resumen de desempeño en test (49,999 observaciones, mismo split para todos).**

| Método | Mejor configuración | AUC-test | F1-test | Pérdida esp. (\$) | Brier | t_fit (s) |
|---|---|---|---|---|---|---|
| Reg. Logística | C=0.10, L2, class_weight, prop=0.80 | **0.6581** | **0.6040** | **994.22** | 0.2110 | **0.05** |
| Árbol | gini, depth=10, ccp_α=0.001, smote | 0.6500 | 0.5589 | 1408.34 | 0.2113 | 0.36 |
| SVM | linear, C=10, class_weight | 0.6500 | 0.5349 | 1055.90 | 0.2140 | 3.92 |
| KNN | k=31, euclidean, uniform | 0.6442 | 0.5452 | 1185.20 | 0.2152 | 0.24 |
| MLP | (21,), tanh, α=1e-3 | **0.6686** | 0.5468 | 1048.96 | **0.2085** | 4.72 |

**Test de DeLong (10 pares de AUCs).** 9 de 10 pares dan diferencias estadísticamente significativas (p<0.05). El único par no significativo es **Árbol vs. SVM** (p = 0.97), lo cual es consistente: ambos se quedan en AUC ≈ 0.65. El MLP es significativamente mejor que los otros cuatro (p<0.0001 contra todos).

**Modelo elegido: Regresión Logística**. Aunque el MLP gana en AUC y Brier por márgenes pequeños, la regresión logística:

1. Tiene la **menor pérdida esperada** ($994.22 vs $1,048 del MLP) bajo la matriz de costos asimétricos $C_{FN}=5{,}000$, $C_{FP}=1{,}000$. Es decir, después de elegir el threshold óptimo de Youden, la logística clasifica mejor los casos donde el costo del error importa.
2. Es **interpretable**: los OR estandarizados se pueden mostrar uno por uno y leer como "una desviación estándar más en `tipo_entrada_woe` multiplica las odds por 1.58".
3. Es **rápida y estable**: 0.05 s para entrenar, gap train/CV < 0.001 (no hay overfitting).
4. **F1-macro 0.604** — la mejor del lote, consecuencia de un threshold bien calibrado y del balance de clases.

**Por qué no los otros:**
- **MLP (21,) tanh** queda como segundo lugar interesante: gana AUC y Brier, pero su pérdida esperada es solo 5% mejor que el SVM y peor que la logística. Justifica explorarlo si hubiera más datos o más variables.
- **Árbol** es útil como herramienta explicativa (los splits sobre `tipo_entrada_woe` y `duracion_min` son fáciles de leer) pero su pérdida esperada es 42% peor que la logística por la rigidez del threshold = 0.554.
- **SVM** y **KNN** quedan al fondo por costo computacional (KNN tarda 8.4 s en predecir 50k filas) y/o desempeño bajo.

**Limitaciones del análisis.**
- Toda la señal predictiva se concentra en pocas variables (`tipo_entrada_woe`, `duracion_min`, `incidente_c4_woe`). El AUC ≈ 0.66 sugiere que las features actuales no son suficientes para una clasificación más precisa.
- `duracion_min` tiene 22% de outliers por IQR; aunque la decisión de no transformar es metodológicamente correcta para árboles/MLG, valdría la pena evaluar variantes en futuras iteraciones.
- Los WoE se calcularon sobre el dataset completo antes del split, lo que puede introducir leve fuga de información. En un pipeline más cuidadoso convendría calcularlos solo dentro de cada fold de CV.

</font>
